<a href="https://colab.research.google.com/github/deepa22-eng/Machine-Learning-2026/blob/main/Deepa_of_Assignment_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 4 - Deadline 3/6/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Deepa Ranabhat

**AI usage statement:**
To undesrtand and interpretate the code I used claude.ai.

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Perform regression using random forest regression model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: use the default values
- Maximum number of features for each tree: use the default values

You should use the log10 transformed passive permeability values as target values.

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance. You should select the metrics you wish to use.

Based on your analysis, which feature set gives the best results for the regression? Does these results fit with the results you obtained in Assignment 3?

#### D) - Optional for 2 points
For the best performing feature set from A), repeat the analysis using the measured passive permeability values in nm/s (i.e., the non log10 transformed values). Do you observe any measurable difference in using the direct values or the log10 transformed values?


In [ ]:
# Download dataset

%%bash
dataset_url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv"
wget ${dataset_url}
ls

2d_features.csv
sample_data


--2026-03-06 21:47:07--  https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/data/2d_features.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10653 (10K) [text/plain]
Saving to: ‘2d_features.csv’

     0K ..........                                            100% 42.2M=0s

2026-03-06 21:47:07 (42.2 MB/s) - ‘2d_features.csv’ saved [10653/10653]



In [ ]:
# Bash script to download all the dataset. Don't worry if you don't understand it
%%bash

url="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/refs/heads/main/outputs/ml_models/"
dataset_filename="model_data.csv"

rm -f ${dataset_filename}

wget ${url}/${dataset_filename} &> /dev/null

ls

2d_features.csv
model_data.csv
sample_data


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
dataset_1 = pd.read_csv("model_data.csv")
dataset_2 = pd.read_csv("2d_features.csv")

In [ ]:
dataset_1

,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),RingAtoms,Halogens,HeteroAtoms,RotBonds (NRotB),AllBonds,RingCount,...,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA),Ensemble_Average_PSA_Chloroform_ANI,Ensemble_Average_Num_IMHB_Chloroform_ANI,Ensemble_Average_RadiusOfGyration_Chloroform_ANI,P_appLog
0,896.999,803.360,0.253521,65,42,0,19,18,71,7,...,0.456522,3,14,2.64400,209.98,710.02,193.71,0.97,5.68,1.491362
1,852.902,743.496,0.191176,62,42,0,19,13,68,7,...,0.395349,3,13,1.76390,217.82,642.18,252.14,0.87,5.24,1.049218
2,850.930,753.592,0.191176,62,42,0,18,13,68,7,...,0.409091,3,12,2.45750,208.59,671.41,248.01,0.67,5.47,0.812913
3,1020.210,896.016,0.276316,71,33,3,20,21,76,6,...,0.470588,3,12,7.70720,186.66,833.34,186.01,0.60,5.84,0.462398
4,1012.704,900.176,0.236842,70,39,1,20,18,76,7,...,0.460000,5,14,7.42946,214.98,785.02,105.63,2.92,5.60,-0.397940
5,837.830,697.696,0.265625,59,34,3,19,17,64,6,...,0.375000,1,12,4.40258,177.04,622.96,204.17,0.29,5.23,1.690196
6,1006.183,872.816,0.266667,70,33,3,20,20,75,6,...,0.460000,3,12,7.31710,186.66,813.34,175.16,0.32,5.41,1.136721
7,1128.150,931.232,0.256098,77,33,9,26,21,82,6,...,0.470588,3,12,8.44280,186.66,833.34,160.07,0.93,5.93,0.505150
8,1022.182,891.056,0.276316,71,33,3,21,21,76,6,...,0.460000,3,13,6.55350,195.89,804.11,197.66,1.64,5.16,1.235528
9,1078.143,902.096,0.253165,74,33,7,24,20,79,6,...,0.460000,3,12,7.80750,186.66,813.34,217.11,0.00,5.63,0.113943


In [ ]:
dataset_2

,Index,Compound,Smiles,P_app AB (nm/s),P_app BA (nm/s),P_app,Molecular Weight (MW),CharVol (characteristic volume),Flexibility (number of rotatable bonds / number of bonds),Number of Heavy Atoms (HA),...,RotBonds (NRotB),AllBonds,RingCount,NumStereo,Fraction of sp3 Carbon Atoms (FSP3),Hydrogen Bond Donors (HBD),Hydrogen Bond Acceptors (HBA),cLogD^7.4,Topological polar surface area (TPSA),Total non-polar surface area (TNSA)
0,1,6a,O=C(C(N1C(C2=CC=CC(NCCOCCOCCOCCC(N(CC3)CCN3C(C...,2.6,370,31.016125,896.999,803.360,0.253521,65.0,...,18.0,71.0,7.0,2.0,0.456522,3.0,14.0,2.64400,209.98,710.02
1,2,6b,O=C(C(N1C(C2=CC=CC(NC(COCCOCC(N(CC3)CCN3C(C=C4...,1.3,96,11.171392,852.902,743.496,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.395349,3.0,13.0,1.76390,217.82,642.18
2,3,6c,O=C(C(N1C(C2=CC=CC(OCC(NCCCCC(N(CC3)CCN3C(C=C4...,1.0,42,6.480741,850.930,753.592,0.191176,62.0,...,13.0,68.0,7.0,2.0,0.409091,3.0,12.0,2.45750,208.59,671.41
3,4,2d,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.5,2,2.900000,1020.210,896.016,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.470588,3.0,12.0,7.70720,186.66,833.34
4,5,4a,CC1=NC(NC2=NC=C(C(=O)NC3=C(C)C=CC=C3Cl)S2)=CC(...,0.2,1,0.400000,1012.704,900.176,0.236842,70.0,...,18.0,76.0,7.0,3.0,0.460000,5.0,14.0,7.42946,214.98,785.02
5,6,2f,CC1(C)C(=O)N(C2=CC=C(C#N)C(C(F)(F)F)=C2)C(=S)N...,17.0,141,49.000000,837.830,697.696,0.265625,59.0,...,17.0,64.0,6.0,1.0,0.375000,1.0,12.0,4.40258,177.04,622.96
6,7,2b,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,13.5,14,13.700000,1006.183,872.816,0.266667,70.0,...,20.0,75.0,6.0,3.0,0.460000,3.0,12.0,7.31710,186.66,813.34
7,8,2c,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,2.6,4,3.200000,1128.150,931.232,0.256098,77.0,...,21.0,82.0,6.0,3.0,0.470588,3.0,12.0,8.44280,186.66,833.34
8,9,2a,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,3.4,86,17.200000,1022.182,891.056,0.276316,71.0,...,21.0,76.0,6.0,3.0,0.460000,3.0,13.0,6.55350,195.89,804.11
9,10,2e,CC1=C(C2=CC=C(CNC(=O)[C@@H]3C[C@@H](O)CN3C(=O)...,0.6,3,1.300000,1078.143,902.096,0.253165,74.0,...,20.0,79.0,6.0,3.0,0.460000,3.0,12.0,7.80750,186.66,813.34


In [ ]:
features_2d = [
    'Molecular Weight (MW)',
    'CharVol (characteristic volume)',
    'Flexibility (number of rotatable bonds / number of bonds)',
    'Number of Heavy Atoms (HA)',
    'RingAtoms',
    'Halogens',
    'HeteroAtoms',
    'RotBonds (NRotB)',
    'AllBonds',
    'RingCount',
    'NumStereo',
    'Fraction of sp3 Carbon Atoms (FSP3)',
    'Hydrogen Bond Donors (HBD)',
    'Hydrogen Bond Acceptors (HBA)',
    'cLogD^7.4',
    'Topological polar surface area (TPSA)',
    'Total non-polar surface area (TNSA)'
]

features_3d = [
    'Ensemble_Average_PSA_Chloroform_ANI',
    'Ensemble_Average_Num_IMHB_Chloroform_ANI',
    'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]

features_combined = features_2d + features_3d

Foe 2D-features

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.model_selection import cross_validate,ShuffleSplit

# Features and target
features = dataset_1[features_2d].values
target_used = dataset_1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


Score: neg_mean_absolute_error
- Training set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.149 +- 0.014
  - Random Splits (100 splits) : -0.162 +- 0.015
- Test set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.542 +- 0.201
  - Random Splits (100 splits) : -0.446 +- 0.121

Score: neg_mean_squared_error
- Training set: neg_mean_squared_error
  - 5-Fold CV                   : -0.039 +- 0.008
  - Random Splits (100 splits) : -0.045 +- 0.010
- Test set: neg_mean_squared_error
  - 5-Fold CV                   : -0.459 +- 0.363
  - Random Splits (100 splits) : -0.329 +- 0.205

Score: neg_root_mean_squared_error
- Training set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.196 +- 0.022
  - Random Splits (100 splits) : -0.211 +- 0.023
- Test set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.632 +- 0.245
  - Random Splits (100 splits) : -0.549 +- 0.164

Score: neg_max_error
- Training set: neg_max_error
  - 5-Fold CV         

For 3D- features

In [ ]:
# Features and target
features = dataset_1[features_3d].values
target_used = dataset_1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


Score: neg_mean_absolute_error
- Training set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.136 +- 0.011
  - Random Splits (100 splits) : -0.142 +- 0.016
- Test set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.425 +- 0.078
  - Random Splits (100 splits) : -0.393 +- 0.101

Score: neg_mean_squared_error
- Training set: neg_mean_squared_error
  - 5-Fold CV                   : -0.033 +- 0.006
  - Random Splits (100 splits) : -0.036 +- 0.008
- Test set: neg_mean_squared_error
  - 5-Fold CV                   : -0.298 +- 0.131
  - Random Splits (100 splits) : -0.260 +- 0.123

Score: neg_root_mean_squared_error
- Training set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.181 +- 0.016
  - Random Splits (100 splits) : -0.188 +- 0.020
- Test set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.532 +- 0.122
  - Random Splits (100 splits) : -0.495 +- 0.123

Score: neg_max_error
- Training set: neg_max_error
  - 5-Fold CV         

3) For Combined features

In [ ]:
# Features and target
features = dataset_1[features_combined].values
target_used = dataset_1['P_appLog'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


Score: neg_mean_absolute_error
- Training set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.129 +- 0.006
  - Random Splits (100 splits) : -0.138 +- 0.017
- Test set: neg_mean_absolute_error
  - 5-Fold CV                   : -0.429 +- 0.188
  - Random Splits (100 splits) : -0.351 +- 0.120

Score: neg_mean_squared_error
- Training set: neg_mean_squared_error
  - 5-Fold CV                   : -0.033 +- 0.005
  - Random Splits (100 splits) : -0.037 +- 0.009
- Test set: neg_mean_squared_error
  - 5-Fold CV                   : -0.329 +- 0.299
  - Random Splits (100 splits) : -0.241 +- 0.172

Score: neg_root_mean_squared_error
- Training set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.182 +- 0.014
  - Random Splits (100 splits) : -0.192 +- 0.024
- Test set: neg_root_mean_squared_error
  - 5-Fold CV                   : -0.522 +- 0.239
  - Random Splits (100 splits) : -0.461 +- 0.168

Score: neg_max_error
- Training set: neg_max_error
  - 5-Fold CV         

Among the three features that I studied , it seems combined feature perform good random forest regression. Here this feature gives lowest mean absolute error (-0.351 +- 0.120) and higher regression (0.125+- 0.508) among 3D and 2D features.

2) Optional part

Repeating the analysis of combined features using the measured passive permeability values in nm/s (i.e., the non log10 transformed values).


In [ ]:
dataset_1['P_app'] = 10**dataset_1['P_appLog']

In [ ]:
dataset_1['P_app']

,P_app
0,31.0
1,11.2
2,6.5
3,2.9
4,0.4
5,49.0
6,13.7
7,3.2
8,17.2
9,1.3


In [ ]:

# Features and target
features = dataset_1[features_combined].values
target_used = dataset_1['P_app'].values

model = RandomForestRegressor(n_estimators=100, random_state=42)


scoring =['neg_mean_absolute_error',
          'neg_mean_squared_error',
          'neg_root_mean_squared_error',
          'neg_max_error',
          'r2']

# employ 5-fold CV
scores_fold = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)

NumSplits=100
cv_random = ShuffleSplit(n_splits=NumSplits, test_size=0.20)
scores_random = cross_validate(
    model,
    features, target_used,
    scoring=scoring,
    cv=cv_random,
    return_train_score=True,
    return_estimator=True,
    return_indices=True
)
for s in scoring:
  print("Score: {:s}".format(s))
  print("- Training set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['train_'+s]),np.nanstd(scores_fold['train_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['train_'+s]), np.nanstd(scores_random['train_'+s])))
  print("- Test set: {:s}".format(s))
  print("  - 5-Fold CV                   : {:.3f} +- {:.3f}".format( np.nanmean(scores_fold['test_'+s]),np.nanstd(scores_fold['test_'+s])))
  print("  - Random Splits ({:d} splits) : {:.3f} +- {:.3f}".format(NumSplits, np.nanmean(scores_random['test_'+s]), np.nanstd(scores_random['test_'+s])))
  print("")


Score: neg_mean_absolute_error
- Training set: neg_mean_absolute_error
  - 5-Fold CV                   : -2.946 +- 0.612
  - Random Splits (100 splits) : -2.787 +- 0.435
- Test set: neg_mean_absolute_error
  - 5-Fold CV                   : -8.064 +- 2.012
  - Random Splits (100 splits) : -7.918 +- 2.213

Score: neg_mean_squared_error
- Training set: neg_mean_squared_error
  - 5-Fold CV                   : -16.233 +- 5.209
  - Random Splits (100 splits) : -15.496 +- 4.133
- Test set: neg_mean_squared_error
  - 5-Fold CV                   : -105.592 +- 48.899
  - Random Splits (100 splits) : -117.986 +- 59.153

Score: neg_root_mean_squared_error
- Training set: neg_root_mean_squared_error
  - 5-Fold CV                   : -3.963 +- 0.727
  - Random Splits (100 splits) : -3.901 +- 0.524
- Test set: neg_root_mean_squared_error
  - 5-Fold CV                   : -9.953 +- 2.557
  - Random Splits (100 splits) : -10.486 +- 2.836

Score: neg_max_error
- Training set: neg_max_error
  - 5-Fold CV

Calculation through the use of passive permeability values in nm/s gives the worst result than the log10 transform. Here the the mean absoult value is far from zero and regression value also decreases from 0.125+-0.120 to -1.227+-4.037.
